For an interview day, the clean pipeline should be:

Frame the task and leakage risks.

Read and clean the data.

Run EDA.

Build a strong baseline.

Improve features.

Evaluate properly.

Explain deployment and next steps.

## My strongest recommendation is this example setup:

Description → TF-IDF word + character n-grams.

PartNumber → character n-grams or hashing.

Count, SumPrice → scaled numeric features.

Model → LightGBM or Logistic Regression / Linear SVM.  

## Here are your results:

Linear SVM dominates with 87.74% test accuracy, significantly beating your previous best USE-based model (81.7%). Logistic Regression also performs strongly at 84.47% test accuracy.

The key improvements came from combining multiple text representations with structured features:

Description word n-grams (2,037 features)

Description character n-grams (2,000 features)

PartNumber character n-grams (1,000 features)

Scaled numeric features (Count, SumPrice)


In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score
from scipy.sparse import hstack, csr_matrix
import re
import warnings
warnings.filterwarnings('ignore')

# Load and prepare data
ds = pd.read_csv('./Probetag/trial_day_booking_code_dataset.csv', index_col=0)

# Fill missing values
ds['Description'] = ds['Description'].fillna('')
ds['PartNumber'] = ds['PartNumber'].fillna('')
ds['Count'] = ds['Count'].fillna(0)
ds['SumPrice'] = ds['SumPrice'].fillna(0)

# Clean text function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

ds['Description_clean'] = ds['Description'].apply(clean_text)
ds['PartNumber_clean'] = ds['PartNumber'].apply(clean_text)

# Remove singleton classes (classes with only 1 sample)
class_counts = ds['BookingCode'].value_counts()
valid_classes = class_counts[class_counts > 1].index
ds_filtered = ds[ds['BookingCode'].isin(valid_classes)].copy()

print(f"Samples after filtering: {len(ds_filtered)}, Classes: {ds_filtered['BookingCode'].nunique()}")

# Train/val/test split
from sklearn.model_selection import train_test_split

X = ds_filtered[['Description_clean', 'PartNumber_clean', 'Count', 'SumPrice']]
y = ds_filtered['BookingCode']

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp)

# Feature extraction
# 1. Description: word n-grams (1-2) with TF-IDF
tfidf_desc_word = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
desc_word_train = tfidf_desc_word.fit_transform(X_train['Description_clean'])
desc_word_val = tfidf_desc_word.transform(X_val['Description_clean'])
desc_word_test = tfidf_desc_word.transform(X_test['Description_clean'])

# 2. Description: character n-grams (2-4) with TF-IDF
tfidf_desc_char = TfidfVectorizer(max_features=2000, analyzer='char', ngram_range=(2, 4), min_df=2, sublinear_tf=True)
desc_char_train = tfidf_desc_char.fit_transform(X_train['Description_clean'])
desc_char_val = tfidf_desc_char.transform(X_val['Description_clean'])
desc_char_test = tfidf_desc_char.transform(X_test['Description_clean'])

# 3. PartNumber: character n-grams (2-5) with TF-IDF
tfidf_part = TfidfVectorizer(max_features=1000, analyzer='char', ngram_range=(2, 5), min_df=2, sublinear_tf=True)
part_train = tfidf_part.fit_transform(X_train['PartNumber_clean'])
part_val = tfidf_part.transform(X_val['PartNumber_clean'])
part_test = tfidf_part.transform(X_test['PartNumber_clean'])

# 4. Numeric features: Count and SumPrice (scaled)
scaler = StandardScaler()
numeric_train = scaler.fit_transform(X_train[['Count', 'SumPrice']])
numeric_val = scaler.transform(X_val[['Count', 'SumPrice']])
numeric_test = scaler.transform(X_test[['Count', 'SumPrice']])

# Combine all features
X_train_combined = hstack([desc_word_train, desc_char_train, part_train, csr_matrix(numeric_train)])
X_val_combined = hstack([desc_word_val, desc_char_val, part_val, csr_matrix(numeric_val)])
X_test_combined = hstack([desc_word_test, desc_char_test, part_test, csr_matrix(numeric_test)])

print(f"\nCombined features: {X_train_combined.shape[1]} total")
print(f"  - Description word n-grams: {desc_word_train.shape[1]}")
print(f"  - Description char n-grams: {desc_char_train.shape[1]}")
print(f"  - PartNumber char n-grams: {part_train.shape[1]}")
print(f"  - Numeric features: 2")

# Train Linear SVM
print("\n" + "="*70)
print("Training Linear SVM...")
print("="*70)

svm_model = LinearSVC(max_iter=2000, random_state=42, C=0.5, class_weight='balanced', dual=False)
svm_model.fit(X_train_combined, y_train)

# Predictions
y_val_pred = svm_model.predict(X_val_combined)
y_test_pred = svm_model.predict(X_test_combined)

# Evaluation
val_acc = accuracy_score(y_val, y_val_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"\nValidation Accuracy: {val_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Detailed classification report on test set
print("\nTest Set Classification Report (top 10 classes by support):")
report = classification_report(y_test, y_test_pred, output_dict=True)
report_df = pd.DataFrame(report).T
report_df = report_df[report_df['support'] > 0].sort_values('support', ascending=False).head(10)
print(report_df[['precision', 'recall', 'f1-score', 'support']].to_string())

Samples after filtering: 3667, Classes: 94

Combined features: 5039 total
  - Description word n-grams: 2037
  - Description char n-grams: 2000
  - PartNumber char n-grams: 1000
  - Numeric features: 2

Training Linear SVM...

Validation Accuracy: 0.8756
Test Accuracy: 0.8774

Test Set Classification Report (top 10 classes by support):
              precision    recall  f1-score  support
macro avg      0.749172  0.767366  0.739700    734.0
weighted avg   0.895370  0.877384  0.878756    734.0
C201           1.000000  0.955224  0.977099     67.0
E416           0.962963  0.852459  0.904348     61.0
P149           0.975610  0.909091  0.941176     44.0
I104           1.000000  0.950000  0.974359     40.0
E326           0.875000  0.636364  0.736842     33.0
E412           0.911765  0.939394  0.925373     33.0
E102           0.966667  0.935484  0.950820     31.0
A101           0.935484  1.000000  0.966667     29.0
